In [10]:
import pandas as pd
import numpy as np
import os
from ukhls_variables import VARIABLE_MAP, DISABILITY_LABELS

# --- PATH CONFIGURATION ---
WAVES = ["o", "n", "m", "l"]  # List of waves to process
RAW_DIR = "../data/ukhls/raw"
PICKLE_DIR = "../data/ukhls/pickles"

def clean_biography_intake():
    if not os.path.exists(PICKLE_DIR): os.makedirs(PICKLE_DIR)

    for w in WAVES:
        ind_file = os.path.join(RAW_DIR, f"{w}_indresp.tab")
        
        if not os.path.exists(ind_file):
            print(f"Skipping {w}: Raw file not found.")
            continue

        print(f"--- Processing Wave {w} Biography ---")
        
        # A. Identify headers and filter targeted columns
        headers = pd.read_csv(ind_file, sep='\t', nrows=0).columns.tolist()
        load_cols = ['pidp']
        for base in VARIABLE_MAP.keys():
            prefixed = f"{w}_{base}"
            if prefixed in headers: load_cols.append(prefixed)
            elif base in headers: load_cols.append(base) # Fallback for pidp
        
        # Add all disability indicators (e.g., o_disdif1, o_disdif2...)
        dis_prefix = f"{w}_disdif"
        dis_raw_cols = [c for c in headers if c.startswith(dis_prefix)]
        load_cols += dis_raw_cols

        # B. Load Data
        df = pd.read_csv(ind_file, sep='\t', usecols=load_cols, low_memory=False)
        df = df.mask(df < 0) # UKHLS missing codes (-9 to -1) -> NaN

        # C. Collapse Disability Multi-Response into a single Category
        print(f"   -> Collapsing {len(dis_raw_cols)} disability columns...")
        def collapse_dis(row):
            for code, label in DISABILITY_LABELS.items():
                col_name = f"{dis_prefix}{code}"
                if col_name in row and row[col_name] == 1:
                    return label
            return "None"

        df[f"{w}_primary_disability"] = df.apply(collapse_dis, axis=1)
        df.drop(columns=dis_raw_cols, inplace=True)

        # D. Clean Industry/Occupation (SIC/SOC) as Strings
        # This prevents "8411" from becoming "8411.0" or an integer
        for job_col in ['jbisic8_dv', 'jbsoc10_dv']:
            full_col = f"{w}_{job_col}"
            if full_col in df.columns:
                print(f"   -> Cleaning {job_col} formatting...")
                df[full_col] = df[full_col].astype(str).replace(['nan', 'NaN', 'None'], np.nan)
                df[full_col] = df[full_col].str.replace(r'\.0$', '', regex=True)

        # E. Final Memory Optimization
        for col in df.columns:
            if 'idp' in col: continue # Preserve PIDP precision
            # If low unique count or string, use category to save RAM
            if df[col].nunique() < 100 or df[col].dtype == 'object':
                df[col] = df[col].astype('category')
            else:
                df[col] = pd.to_numeric(df[col], downcast='float')

        # F. Save Optimized Pickle
        out_path = os.path.join(PICKLE_DIR, f"{w}_indresp_optimized.pkl")
        df.to_pickle(out_path, protocol=5)
        print(f"   Done. Saved {len(df.columns)} biography columns for {len(df):,} rows.\n")

if __name__ == "__main__":
    clean_biography_intake()

--- Processing Wave o Biography ---
   -> Collapsing 13 disability columns...
   Done. Saved 9 biography columns for 32,849 rows.

--- Processing Wave n Biography ---
   -> Collapsing 13 disability columns...
   Done. Saved 9 biography columns for 35,471 rows.

--- Processing Wave m Biography ---
   -> Collapsing 13 disability columns...
   Done. Saved 9 biography columns for 27,998 rows.

--- Processing Wave l Biography ---
   -> Collapsing 13 disability columns...
   Done. Saved 9 biography columns for 29,271 rows.

